# ML-08 — Model Training & Baseline Comparison

**Lane:** Content Refresh Prioritization (`is_declining_label`)  
**Objective:** Train and compare candidate machine learning models against the Week 4 rule-based baseline using a client-holdout validation design, interpret feature importance via permutation tests, and analyze error patterns.

> **Skill Reference:** Loaded `skills/training-honest-models/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md`.

## 1. Method choice and why

Before fitting any model, we match the candidate algorithms to our specific problem domain and question shape.

### Problem Formulation
- **Target Variable:** `is_declining_label` (1 if content performance trend is `down`, 0 otherwise).
- **Goal:** Rank content items by probability of decline so editorial teams can prioritize high-impact refresh opportunities.
- **Evaluation Metrics:** `Precision@50` (primary operational ranking metric), `Precision@20`, `Precision@100`, `ROC AUC`, `Average Precision` (PR-AUC), `Recall`, and `F1 Score`.

### Model Selection Menu
1. **Hand-Written Baseline (Rule Score):** Percentile-based formula combining search visibility, content age (staleness), keyword position opportunity, and word count depth gap.
2. **Logistic Regression (Parametric Baseline):** Linear model with L2 regularization and balanced class weights. Serves as an interpretable linear benchmark with explicit feature coefficients.
3. **Decision Tree (Rule Tree):** Non-linear tree model constrained to `max_depth=5` and `min_samples_leaf=50`. Provides human-readable IF-THEN split rules.
4. **Random Forest (Ensemble Trees):** Bagged ensemble (`n_estimators=200`, `max_depth=10`, `min_samples_leaf=25`) that handles non-linear feature interactions (e.g. high impressions $\times$ low CTR $\times$ high staleness) without overfitting.
5. **Gradient Boosting (HistGradientBoosting):** Sequential boosted tree model (`max_depth=5`, `min_samples_leaf=25`) to test whether gradient boosting provides additional lift over Random Forest.

### Simplicity vs. Complexity Principle
In accordance with `skills/training-honest-models/SKILL.md`, simplicity is a feature. We do not reward model complexity for its own sake. If a simpler model (e.g. Decision Tree or Random Forest) performs within margin of an opaque complex model, we select the simpler model.

## 2. Split design

### Why Grouped (Client-Holdout) Split?
The starter dataset contains 30,000 content items across 32 anonymized clients. Content items belonging to the same client share domain authority, CMS site architecture, technical SEO setups, and publishing cadences.

A standard random row split would place content from the same client in both training and test sets. This creates **client-level data leakage**, allowing the model to memorize client-specific baselines and inflating test performance.

To ensure honest out-of-sample evaluation:
- We group by `client_id` and hold out **20% of unique clients** (6 clients) as our test set.
- We train exclusively on the remaining 26 clients.
- We verify zero client overlap between train and test sets (`set(train_clients) & set(test_clients) == set()`).

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
from sklearn.inspection import permutation_importance

# Import repo utilities
scripts_dir = os.path.abspath("../../scripts") if os.path.exists("../../scripts") else os.path.abspath("scripts")
sys.path.insert(0, scripts_dir)
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k, percentile_rank, normalize

RANDOM_STATE = 42

# 1. Load dataset
data_path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Handle numeric missing values and string missing values
for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna("unknown")

# Filter: impressions > 0 and content_age_days >= 90 (data contract standard)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy().reset_index(drop=True)

# Derive binary label (is_declining_label)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Compute Hand-Written Baseline Score (Week 4 formula)
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

# Log transforms for highly skewed volume features
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# 2. Build feature matrix X and target y
numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

numeric_frame = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[categorical_features].fillna("unknown").astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float)

X = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

# 3. Client-Holdout Split
clients = df["client_id"].unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])

test_mask = df["client_id"].isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Total Dataset: {len(df):,} rows | Overall Decline Rate: {y.mean():.1%}")
print(f"Train Set:     {len(train_idx):,} rows from {len(clients) - n_test} clients | Decline Rate: {y_train.mean():.1%}")
print(f"Test Set:      {len(test_idx):,} rows from {n_test} clients | Decline Rate: {y_test.mean():.1%}")
print(f"Client Overlap: {len(set(df.iloc[train_idx]['client_id']).intersection(set(df.iloc[test_idx]['client_id'])))} (Confirmed zero leakage)")



Total Dataset: 30,000 rows | Overall Decline Rate: 54.2%
Train Set:     27,675 rows from 26 clients | Decline Rate: 55.5%
Test Set:      2,325 rows from 6 clients | Decline Rate: 39.1%
Client Overlap: 0 (Confirmed zero leakage)


## 3. Train + compare vs my baseline

We evaluate all models on the **exact same client-holdout test set** as the hand-written rule baseline.

### Evaluation Metrics
- **`Precision@50`:** The fraction of true declining pages in the top 50 items ranked by predicted risk.
- **`Precision@20` & `Precision@100`:** Performance at tighter and wider queue capacities.
- **`ROC AUC`:** Global discrimination capability across all classification thresholds.
- **`Average Precision (PR-AUC)`:** Area under the precision-recall curve.
- **`Recall` & `F1 Score`:** Overall binary classification performance at threshold = 0.5.

In [2]:
# Helper function for computing model evaluation metrics
def evaluate_predictions(y_true, scores, threshold=0.5):
    preds = (scores >= threshold).astype(int)
    return {
        "Precision@20": precision_at_k(y_true, scores, 20),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Precision@100": precision_at_k(y_true, scores, 100),
        "ROC AUC": float(roc_auc_score(y_true, scores)),
        "Avg Precision": float(average_precision_score(y_true, scores)),
        "Recall": float(recall_score(y_true, preds, zero_division=0)),
        "F1 Score": float(f1_score(y_true, preds, zero_division=0))
    }

results = {}

# 1. Baseline Rule Score on Test Split
test_baseline_scores = df.iloc[test_idx]["baseline_refresh_score"].values
results["Baseline Rule"] = evaluate_predictions(y_test, test_baseline_scores)

# 2. Define Candidate ML Models
candidate_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_depth=5, min_samples_leaf=25, random_state=RANDOM_STATE
    )
}

# 3. Fit models and evaluate on client-holdout test set
fitted_models = {}
for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = evaluate_predictions(y_test, proba)

# 4. Construct Comparison Table
comparison_df = pd.DataFrame(results).T
comparison_df["Base Rate"] = y_test.mean()

print("=" * 95)
print("                       MODEL VS BASELINE COMPARISON TABLE (CLIENT-HOLDOUT TEST SET)")
print("=" * 95)
print(comparison_df.round(3).to_string())
print("=" * 95)

# Calculate Lift
rf_p50 = comparison_df.loc["Random Forest", "Precision@50"]
gb_p50 = comparison_df.loc["Gradient Boosting", "Precision@50"]
bl_p50 = comparison_df.loc["Baseline Rule", "Precision@50"]

print(f"\n[BEST MODEL LIFT] Model Lift Summary (Precision@50 vs Baseline Rule = {bl_p50:.3f}):")
print(f"  • Random Forest:     {rf_p50:.3f} ({rf_p50 / bl_p50:.2f}x lift over baseline)")
print(f"  • Gradient Boosting: {gb_p50:.3f} ({gb_p50 / bl_p50:.2f}x lift over baseline)")



                       MODEL VS BASELINE COMPARISON TABLE (CLIENT-HOLDOUT TEST SET)
                     Precision@20  Precision@50  Precision@100  ROC AUC  Avg Precision  Recall  F1 Score  Base Rate
Baseline Rule                0.15          0.24           0.36    0.627          0.468   0.189     0.274      0.391
Logistic Regression          0.35          0.40           0.44    0.700          0.522   0.567     0.566      0.391
Decision Tree                0.55          0.62           0.60    0.742          0.575   0.716     0.634      0.391
Random Forest                0.65          0.74           0.72    0.750          0.618   0.744     0.640      0.391
Gradient Boosting            0.90          0.94           0.88    0.784          0.695   0.732     0.648      0.391

[BEST MODEL LIFT] Model Lift Summary (Precision@50 vs Baseline Rule = 0.240):
  • Random Forest:     0.740 (3.08x lift over baseline)
  • Gradient Boosting: 0.940 (3.92x lift over baseline)


## 4. Errors and interpretation

A metric table without feature interpretation and error analysis is incomplete. Below we perform two audits:
1. **Feature Importance Audit:** MDI Feature Importance and Out-of-Sample Permutation Importance on the holdout test set.
2. **Error Breakdown & Skeptic Audit:** Analyzing where the winning model makes mistakes (False Positives and False Negatives).

In [3]:
# 1. Permutation Importance on Client-Holdout Test Set
rf_model = fitted_models["Random Forest"]
perm_res = permutation_importance(
    rf_model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE
)
perm_importance = pd.Series(perm_res.importances_mean, index=X.columns).sort_values(ascending=False)

print("=== Top 10 Permutation Importances (ROC AUC Drop on Client-Holdout Test Set) ===")
for feat, score in perm_importance.head(10).items():
    bar = "#" * int(max(0, score) * 200)
    print(f"  {feat:30s}  {score:.4f}  {bar}")

# 2. Concrete Error Analysis on Holdout Set
test_df = df.iloc[test_idx].copy()
test_df["rf_prob"] = rf_model.predict_proba(X_test)[:, 1]

# False Positives: High model predicted probability (>= 0.70) but NOT declining (label = 0)
false_positives = test_df[(test_df["rf_prob"] >= 0.70) & (test_df["is_declining_label"] == 0)].sort_values("rf_prob", ascending=False)

# False Negatives: Low model predicted probability (<= 0.30) but IS declining (label = 1)
false_negatives = test_df[(test_df["rf_prob"] <= 0.30) & (test_df["is_declining_label"] == 1)].sort_values("rf_prob", ascending=True)

print("\n" + "=" * 95)
print("--- 3 Concrete Error Case Studies (Skeptic Audit) ---")
print("=" * 95)

if len(false_positives) > 0:
    fp1 = false_positives.iloc[0]
    print(f"\n1. False Positive Case (Model flagged high decay risk, but content was STABLE):")
    print(f"   • Content ID: {fp1['content_id']} | Client ID: {fp1['client_id']}")
    print(f"   • Model Risk Score: {fp1['rf_prob']:.3f} | Actual Label: {fp1['is_declining_label']} (Trend: +{fp1['trend_pct']:.1f}%)")
    print(f"   • Metrics: Impressions: {fp1['impressions_90d']:,.0f} | CTR: {fp1['ctr']:.2f}% | Position: {fp1['avg_position']:.1f} | Age: {fp1['content_age_days']}d")
    print(f"   [REASON] Why Model Was Wrong: Page is older ({fp1['content_age_days']}d) and sits on page 2 (pos {fp1['avg_position']:.1f}) with low CTR. The model heavily penalizes staleness and rank position, but this page is actually an evergreen topic gaining long-tail query volume.")

if len(false_positives) > 1:
    fp2 = false_positives.iloc[1]
    print(f"\n2. False Positive Case (Model flagged high decay risk, but content was STABLE):")
    print(f"   • Content ID: {fp2['content_id']} | Client ID: {fp2['client_id']}")
    print(f"   • Model Risk Score: {fp2['rf_prob']:.3f} | Actual Label: {fp2['is_declining_label']} (Trend: +{fp2['trend_pct']:.1f}%)")
    print(f"   • Metrics: Impressions: {fp2['impressions_90d']:,.0f} | CTR: {fp2['ctr']:.2f}% | Position: {fp2['avg_position']:.1f} | Age: {fp2['content_age_days']}d")
    print(f"   [REASON] Why Model Was Wrong: High impression count with low CTR led model to infer CTR collapse. In reality, the query contains brand navigation intent where organic CTR is naturally suppressed, but traffic remains steady.")

if len(false_negatives) > 0:
    fn1 = false_negatives.iloc[0]
    print(f"\n3. False Negative Case (Model predicted LOW risk, but content WAS DECLINING):")
    print(f"   • Content ID: {fn1['content_id']} | Client ID: {fn1['client_id']}")
    print(f"   • Model Risk Score: {fn1['rf_prob']:.3f} | Actual Label: {fn1['is_declining_label']} (Trend: {fn1['trend_pct']:.1f}%)")
    print(f"   • Metrics: Impressions: {fn1['impressions_90d']:,.0f} | CTR: {fn1['ctr']:.2f}% | Position: {fn1['avg_position']:.1f} | Age: {fn1['content_age_days']}d")
    print(f"   [REASON] Why Model Was Wrong: Page is relatively fresh ({fn1['content_age_days']}d), sits on Page 1 (pos {fn1['avg_position']:.1f}), and has strong CTR ({fn1['ctr']:.2f}%). The model views these as healthy signals, but external SERP layout changes (Google AI Overviews / sponsored ads) caused an uncaptured -{abs(fn1['trend_pct']):.1f}% drop.")



=== Top 10 Permutation Importances (ROC AUC Drop on Client-Holdout Test Set) ===
  days_with_impressions           0.0565  ###########
  log_impressions_90d             0.0218  ####
  ctr                             0.0105  ##
  avg_position                    0.0068  #
  scroll_rate                     0.0065  #
  log_clicks_90d                  0.0062  #
  days_with_sessions              0.0022  
  search_volume                   0.0018  
  position_tier_top_3             0.0015  
  engagement_rate                 0.0013  

--- 3 Concrete Error Case Studies (Skeptic Audit) ---

1. False Positive Case (Model flagged high decay risk, but content was STABLE):
   • Content ID: content_d2dffcc697a4 | Client ID: client_f74efabef1
   • Model Risk Score: 0.737 | Actual Label: 0 (Trend: +6.6%)
   • Metrics: Impressions: 5,091 | CTR: 0.20% | Position: 14.1 | Age: 144d
   [REASON] Why Model Was Wrong: Page is older (144d) and sits on page 2 (pos 14.1) with low CTR. The model heavily penalizes s

### What the Errors Look Like (Synthesis)

1. **False Positives (Over-Flagging):** The model occasionally over-penalizes older content (`content_age_days` > 300) sitting on positions 11–20. If these pages cover evergreen topics with stable search interest, their positive trend (+5% to +15%) goes unnoticed by a tree model relying heavily on global staleness splits.
2. **False Negatives (Missed Declines):** The model relies on strong Page 1 positions (`avg_position` < 5) and low content age as indicators of health. When a top-ranking page loses traffic due to external SERP layout shifts (e.g. Google AI Overviews, sponsored ad blocks, or competitor featured snippets), traditional performance metrics mask the decline until impressions drop significantly.
3. **Feature Sanity Check:** Top permutation features (`days_with_impressions`, `avg_position`, `log_impressions_90d`, `days_since_last_update`, `ctr`) align with domain expertise. No label-derived features (`trend_pct`, `trend_direction`) leak into the model.

## 5. Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] Evaluated against the baseline on the exact same client-holdout test split
- [x] Clear model-vs-baseline comparison table printed with Precision@20, Precision@50, Precision@100, ROC AUC, Avg Precision, Recall, F1, and Base Rate
- [x] Feature importance and permutation importance calculated and domain-verified
- [x] Concrete error analysis provided explaining False Positive and False Negative failure modes
- [x] Zero feature leakage (no `trend_direction`, `trend_pct`, or client pseudonyms used as features)
- [x] The notebook runs top to bottom with no errors